In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
from tqdm import trange

# Directory containing the TIFF files
directory = r'G:\Hangkai\CONUS Forest Edge Mapping\Forest_Depth_Classification'

# Initialize an empty list to store the results
results = []

# Loop through each year from 2001 to 2019
for year in trange(2001, 2019 + 1):
    file_path = os.path.join(directory, f'CONUS_Edge_Depth_{year}.tif')
    
    # Open the TIFF file
    with rasterio.open(file_path) as src:
        data = src.read(1)
    
    # Count the number of pixels for each depth category (1 to 5)
    pixel_counts = {}
    for depth in range(1, 6):  # Adjust the range if depths are only 1 to 4
        pixel_counts[depth] = np.sum(data == depth)
        print(depth, pixel_counts[depth])
    
    # Calculate the total number of valid pixels
    total_pixels = sum(pixel_counts.values())
    
    # Calculate the proportion for each depth category
    proportions = {depth: count / total_pixels for depth, count in pixel_counts.items()}
    
    # Store the results
    results.append({
        'Year': year,
        **proportions
    })

# Convert results to a DataFrame
df = pd.DataFrame(results)

# Fill missing depth categories with zeros
for depth in range(1, 6): 
    if depth not in df.columns:
        df[depth] = 0.0

# Sort columns
df = df[['Year'] + list(range(1, 6))]

# Save the results to a CSV file
output_file = os.path.join(directory, 'forest_depth_proportions.csv')
df.to_csv(output_file, index=False)

print(f'Results saved to {output_file}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Provided data
data = {
    "Year": list(range(2001, 2020)),
    "Depth 1": [
        678003344, 675708584, 673588358, 674226691, 675390479, 678435485, 681215120, 682897175,
        683580718, 684640451, 687501236, 691405045, 693633594, 695222648, 699860818, 704678471,
        708622684, 713273544, 723409228
    ],
    "Depth 2": [
        414015472, 413753722, 413822369, 414230547, 414723716, 416168409, 417444905, 418411683,
        418501260, 418467744, 419301821, 420633798, 421326066, 421044691, 421815732, 423409626,
        425147661, 427167749, 430028405
    ],
    "Depth 3": [
        296560554, 296679217, 297243540, 297625061, 297891329, 298594520, 299199910, 299712896,
        299976544, 300044856, 300457424, 300992898, 301328978, 301051359, 301148637, 301609509,
        302257028, 302891029, 303590523
    ],
    "Depth 4": [
        227553033, 227824715, 228457893, 228740977, 228837747, 229086109, 229269022, 229439278,
        229763029, 229889228, 230078407, 230193117, 230268038, 230082296, 229915677, 229816713,
        229843236, 229601557, 229048006
    ],
    "Depth 5": [
        1190941485, 1197182666, 1197672398, 1191364490, 1184205162, 1170689578, 1159415107,
        1151023239, 1150644366, 1150973028, 1143833869, 1137247413, 1132507850, 1134345657,
        1129170705, 1118482591, 1107080382, 1086000562, 1057851839
    ]
}

# Create a DataFrame
df = pd.DataFrame(data)

# Set the Year column as the index
df.set_index('Year', inplace=True)

# Plot the proportions for each depth category
for depth in df.columns:
    plt.figure(figsize=(4, 2))
    plt.plot(df.index, df[depth], marker='o', label=depth)
    plt.xlabel('Year')
    plt.ylabel('Pixel Count')
    plt.legend(title=f'Pixel count')


plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Provided data
data = {
    "Year": list(range(2001, 2020)),
    "0-30": [
        678003344, 675708584, 673588358, 674226691, 675390479, 678435485, 681215120, 682897175,
        683580718, 684640451, 687501236, 691405045, 693633594, 695222648, 699860818, 704678471,
        708622684, 713273544, 723409228
    ],
    "0-30": [
        414015472, 413753722, 413822369, 414230547, 414723716, 416168409, 417444905, 418411683,
        418501260, 418467744, 419301821, 420633798, 421326066, 421044691, 421815732, 423409626,
        425147661, 427167749, 430028405
    ],
    "60-90": [
        296560554, 296679217, 297243540, 297625061, 297891329, 298594520, 299199910, 299712896,
        299976544, 300044856, 300457424, 300992898, 301328978, 301051359, 301148637, 301609509,
        302257028, 302891029, 303590523
    ],
    "90-120": [
        227553033, 227824715, 228457893, 228740977, 228837747, 229086109, 229269022, 229439278,
        229763029, 229889228, 230078407, 230193117, 230268038, 230082296, 229915677, 229816713,
        229843236, 229601557, 229048006
    ],
    ">120": [
        1190941485, 1197182666, 1197672398, 1191364490, 1184205162, 1170689578, 1159415107,
        1151023239, 1150644366, 1150973028, 1143833869, 1137247413, 1132507850, 1134345657,
        1129170705, 1118482591, 1107080382, 1086000562, 1057851839
    ]
}

# Create a DataFrame
df = pd.DataFrame(data)

# Calculate total pixels for each year
df['Total'] = df.sum(axis=1)

# Calculate proportions for each depth category
for depth in ['0-30', '0-30', '60-90', '90-120', '>120']:
    df[f'{depth} Proportion'] = df[depth] / df['Total']

# Set the Year column as the index
df.set_index('Year', inplace=True)

# Prepare data for stacked area plot
proportions = df[[f'{depth} Proportion' for depth in ['0-30', '0-30', '60-90', '90-120', '>120']]]

# Plot the stacked area plot
plt.figure(figsize=(12, 8))
proportions.plot(kind='area', stacked=True, figsize=(12, 8), cmap='viridis')

# Add title and labels
plt.title('Area Proportion of Forest at Different Depth Categories (2001-2019)')
plt.xlabel('Year')
plt.ylabel('Proportion')
plt.legend(title='Depth Category')

# Show the plot
plt.tight_layout()
plt.show()